In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Get a summary of the dataset
print(train_data.info())

# Check for missing values
print(train_data.isnull().sum())

# Describe the dataset
print(train_data.describe())

# Distinguish column types
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Visualize the distribution of the target variable
sns.countplot(x='target', data=train_data)
plt.title('Distribution of Target Variable')
plt.show()

# Visualize the correlation matrix for numeric columns
plt.figure(figsize=(12, 8))
correlation_matrix = train_data[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Features')
plt.show()

# Visualize the distribution of numeric features
train_data[numeric_cols].hist(bins=20, figsize=(15, 10))
plt.suptitle('Distribution of Numeric Features')
plt.show()

# Visualize the distribution of categorical features
for col in categorical_cols:
    sns.countplot(x=col, data=train_data)
    plt.title(f'Distribution of {col}')
    plt.show()


    id  gravity    ph  osmo  cond  urea   calc  target
0  192    1.012  5.77   461  17.4   195   1.40       0
1  234    1.017  5.71   704  24.5   270   3.46       0
2    5    1.025  6.90   947  28.4   395   2.64       1
3   45    1.008  5.98   779  17.8   418   6.99       1
4  245    1.031  5.24   703  23.6   364  12.68       1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 331 entries, 0 to 330
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       331 non-null    int64  
 1   gravity  331 non-null    float64
 2   ph       331 non-null    float64
 3   osmo     331 non-null    int64  
 4   cond     331 non-null    float64
 5   urea     331 non-null    int64  
 6   calc     331 non-null    float64
 7   target   331 non-null    int64  
dtypes: float64(4), int64(4)
memory usage: 20.8 KB
None
id         0
gravity    0
ph         0
osmo       0
cond       0
urea       0
calc       0
target     0
dtype: int64
             

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-14 23:54:26.588 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': [], 'Numeric': ['id', 'gravity', 'ph', 'osmo', 'cond', 'urea', 'calc', 'target'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, MinMaxScale

# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/test.csv')

# Preprocess the train data
train_data_copy = train_data.copy()
test_data_copy = test_data.copy()

# Handle missing values
numeric_cols = train_data_copy.select_dtypes(include=[np.number]).columns
fill_missing = FillMissingValue(features=numeric_cols, strategy='mean')
train_data_copy = fill_missing.fit_transform(train_data_copy)
test_data_copy = fill_missing.transform(test_data_copy)

# Normalize numerical features
min_max_scale = MinMaxScale(features=numeric_cols)
train_data_copy = min_max_scale.fit_transform(train_data_copy)
test_data_copy = min_max_scale.transform(test_data_copy)

# Display the preprocessed data
print(train_data_copy.head())
print(test_data_copy.head())


         id   gravity        ph      osmo      cond      urea      calc  target
0  0.463592  0.200000  0.317610  0.261201  0.373860  0.303279  0.086803     0.0
1  0.565534  0.342857  0.298742  0.492850  0.589666  0.426230  0.232181     0.0
2  0.009709  0.571429  0.672956  0.724500  0.708207  0.631148  0.174312     1.0
3  0.106796  0.085714  0.383648  0.564347  0.386018  0.668852  0.481299     1.0
4  0.592233  0.742857  0.150943  0.491897  0.562310  0.580328  0.882851     1.0
         id   gravity        ph      osmo      cond      urea      calc  target
0  0.866505  0.571429  0.399371  0.733079  0.683891  0.759016  0.650670     1.0
1  0.847087  0.457143  0.103774  0.654909  0.726444  0.609836  0.481299     1.0
2  0.902913  0.457143  0.141509  0.512869  0.495441  0.709836  0.237121     1.0
3  0.966019  0.342857  0.566038  0.354623  0.325228  0.503279  0.367678     1.0
4  0.893204  0.171429  0.638365  0.168732  0.316109  0.244262  0.174312     0.0


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data_copy)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'gravity', 'ph', 'osmo', 'cond', 'urea', 'calc', 'target'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Assuming train_data_copy and test_data_copy are already preprocessed
X_train = train_data_copy.drop(columns=['id', 'target'])
y_train = train_data_copy['target']
X_test = test_data_copy.drop(columns=['id'])

# Initialize and train the RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict probabilities on the test set
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate AUC-ROC score
auc_roc = roc_auc_score(test_data['target'], y_pred_proba)
print(f'AUC-ROC Score: {auc_roc}')


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- target
